#### templates.py
```python

import os

# Run this inside your existing project root directory

STRUCTURE = [
    "collector",
    "processor",
    "storage/raw",
    "storage/candles",
    "storage/features",
    "utils",
    "config",
    "app",
    "docker",
]

FILES = {
    "main.py": "",
    "collector/stream.py": "",
    "processor/candles.py": "",
    "processor/features.py": "",
    "storage/writer.py": "",
    "utils/time_utils.py": "",
    "config/settings.py": "",
    "app/dashboard.py": "",
    "docker/docker-compose.yml": "",
    "README.md": "# BTC Intraday ML Pipeline\n",
}


def create_structure(base_path="."):
    print("\n🚀 Initializing project structure...\n")

    # create directories
    for folder in STRUCTURE:
        path = os.path.join(base_path, folder)
        os.makedirs(path, exist_ok=True)
        print(f"📁 {path}")

    # create files
    for file_path, content in FILES.items():
        full_path = os.path.join(base_path, file_path)

        os.makedirs(os.path.dirname(full_path), exist_ok=True)

        if not os.path.exists(full_path):
            with open(full_path, "w") as f:
                f.write(content)
            print(f"📄 created: {full_path}")
        else:
            print(f"⚠️ exists: {full_path} (skipped)")

    print("\n✅ Structure ready inside existing root folder!")


if __name__ == "__main__":
    create_structure()
```

____

```raw
collector/
processor/
storage/
utils/
config/
app/
docker/
main.py
```

___

- fill only these
```raw
collector/stream.py
storage/writer.py
config/settings.py
main.py
```

___

`config/settings.py`

```python
WS_URL = "wss://stream.binance.com:9443/ws/btcusdt@trade"

SYMBOL = "BTCUSDT"

RAW_STORAGE_PATH = "storage/raw/trades.parquet"

BUFFER_SIZE = 10000
```

___

`storage/writer`
```python
import pandas as pd
import os


def save_raw_trades(buffer, path):
    """
    Save list of trade dicts into parquet file.
    Appends if file exists.
    """

    if len(buffer) == 0:
        return

    df = pd.DataFrame(buffer)

    os.makedirs(os.path.dirname(path), exist_ok=True)

    if os.path.exists(path):
        old = pd.read_parquet(path)
        df = pd.concat([old, df], ignore_index=True)

    df.to_parquet(path, index=False)
```

___

`collector/stream.py`
```python
import asyncio
import json
import websockets
from collections import deque

from config.settings import WS_URL, BUFFER_SIZE
from storage.writer import save_raw_trades


class TradeCollector:
    def __init__(self):
        self.buffer = deque(maxlen=BUFFER_SIZE)

    async def connect(self):
        async with websockets.connect(WS_URL, ping_interval=20) as ws:
            print("✅ Connected to Binance stream")

            while True:
                msg = await ws.recv()
                data = json.loads(msg)

                trade = self.normalize(data)

                self.buffer.append(trade)

                # persist in batches (important for big data systems)
                if len(self.buffer) % 200 == 0:
                    save_raw_trades(list(self.buffer), "storage/raw/trades.parquet")

                print(trade)

    def normalize(self, data):
        """
        Convert raw Binance message → structured schema
        """

        return {
            "timestamp": data["T"],
            "price": float(data["p"]),
            "quantity": float(data["q"]),
            "is_buyer_maker": data["m"],
            "trade_id": data["t"]
        }
```

___

`main.py`

```python
import asyncio
from collector.stream import TradeCollector


async def run():
    collector = TradeCollector()
    await collector.connect()


if __name__ == "__main__":
    asyncio.run(run())
```
